## Text Preprocessing and Vectorization

### Objective

The objective of this step is to prepare customer complaint text for analysis by applying simple and efficient preprocessing and converting the text into numerical features using TF-IDF. The goal is to make the data suitable for topic modeling and trend analysis while keeping the workflow easy to understand and computationally manageable.


In [13]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import pickle

In [3]:
df = pd.read_csv("../data/processed/complaints_step1_filtered.csv")

df = df.sample(n=50000, random_state=42).reset_index(drop=True)

df.head(2)

,Date received,Consumer complaint narrative,Product,Issue,Sub-issue,Company
0,2018-07-20,I live in the city of XXXX Maryland. The City ...,Mortgage,Trouble during payment process,NaN,PRIMARY RESIDENTIAL MORTGAGE
1,2017-06-23,My account balance on my capital one checking ...,Checking or savings account,Managing an account,Funds not handled or disbursed as instructed,CAPITAL ONE FINANCIAL CORPORATION


In [4]:
df["clean_text"] = (
    df["Consumer complaint narrative"]
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [5]:
df[["Consumer complaint narrative", "clean_text"]].head(3)

,Consumer complaint narrative,clean_text
0,I live in the city of XXXX Maryland. The City ...,i live in the city of xxxx maryland. the city ...
1,My account balance on my capital one checking ...,my account balance on my capital one checking ...
2,multiple attempts to collect on a debt that ha...,multiple attempts to collect on a debt that ha...


In [14]:
import re

def remove_masked_tokens(text):
    text = re.sub(r"\b[xX]{2,}\b", " ", text)      # xx, xxxx
    text = re.sub(r"\b0{2,}\b", " ", text)         # 00, 000
    text = re.sub(r"\bxx/xx/xxxx\b", " ", text)    # masked dates
    return text

df["clean_text"] = df["clean_text"].apply(remove_masked_tokens)

In [15]:
df["clean_text"].head(5)

0    i live in the city of   maryland. the city off...
1    my account balance on my capital one checking ...
2    multiple attempts to collect on a debt that ha...
3    i have informed transunion that the alabama   ...
4    i pulled my credit report on  / /2018 and noti...
Name: clean_text, dtype: object

In [16]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_df=0.95,
    min_df=10,
    ngram_range=(1, 2)
)

X_tfidf = vectorizer.fit_transform(df["clean_text"])

In [17]:
X_tfidf.shape

(50000, 57101)

In [18]:
df.to_csv("../data/processed/complaints_step2_cleaned.csv", index=False)

with open("../data/processed/tfidf_matrix.pkl", "wb") as f:
    pickle.dump(X_tfidf, f)

with open("../data/processed/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

### Preprocessing Approach and Design Choice

In this project, the focus is on identifying recurring issues and trends from customer complaints rather than performing detailed linguistic analysis. While libraries such as spaCy or NLTK can be used for advanced text preprocessing, they were not used here due to computational limitations and the scale of the dataset.

Instead, a lightweight preprocessing approach was adopted:
- Applied minimal text normalization (lowercasing and whitespace cleanup) to keep the original meaning intact.
- Avoided heavy linguistic steps such as lemmatization to ensure the workflow remained efficient and runnable on limited hardware.
- Performed tokenization, stopword removal, and feature filtering during the TF-IDF vectorization step.

This approach allowed the project to focus on learning core NLP concepts and extracting meaningful insights without overcomplicating the preprocessing pipeline.

### Refinement Based on Initial Topic Modeling

After running an initial round of topic modeling, some topics were dominated by masked or redacted tokens such as "xxxx", "xx", and placeholder numbers. These tokens do not carry meaningful information about customer issues and were affecting topic clarity.

To improve interpretability:
- Masked alphabetic and numeric placeholders were removed from the cleaned text.
- No real numbers, dates, or financial amounts were removed.
- The change was intentionally limited to avoid altering the original meaning of complaints.

After this refinement, the TF-IDF feature space was slightly reduced, indicating that non-informative tokens were successfully removed while preserving meaningful vocabulary. This helped produce clearer and more interpretable topics in the subsequent modeling step.

### Summary

- Loaded the cleaned customer complaint dataset from the previous step.
- Applied basic text normalization to improve consistency while preserving meaning.
- Removed masked and redacted tokens that were affecting topic quality.
- Avoided heavy linguistic preprocessing due to computational constraints and learning focus.
- Converted complaint text into numerical features using TF-IDF vectorization.
- Prepared the refined data for topic modeling and trend analysis.